In [26]:
import os
import re
import csv
import pandas as pd
from pathlib import Path
from lxml import etree

In [27]:
DATA_DIR          = "../data"            
LOOKUP_CSV        = "../church-fathers.csv" 
OUTPUT_CSV        = "sources_catalogue.csv"  
LOOKUP_SEP        = ";"   

LOOKUP_KEY_COL    = "CF_ID"               
LOOKUP_NAME_COL   = "Name"     

In [28]:

lookup_path = Path(LOOKUP_CSV)
lookup_df = pd.read_csv(lookup_path, sep=LOOKUP_SEP, dtype=str)
lookup_df.columns = lookup_df.columns.str.strip()

# Normalise the key column 
lookup_df[LOOKUP_KEY_COL] = lookup_df[LOOKUP_KEY_COL].str.strip().str.lower()

lookup = dict(zip(lookup_df[LOOKUP_KEY_COL], lookup_df[LOOKUP_NAME_COL]))

print(f"Loaded {len(lookup)} entries from '{LOOKUP_CSV}'")
print(lookup_df.head())

Loaded 53 entries from '../church-fathers.csv'
   CF_ID bullinger_ID                     Name     gnd_id cc_id   class  \
0  cf001       P18881  Ignatius von Antiochien  118555340   NaN  syrian   
1  cf002       P19930      Polycarpus Smyrnäus  11859558X   NaN   greek   
2  cf003       P19894  Athenagoras Atheniensis  118646141   NaN   greek   
3  cf004       P21365   Theophilus Antiochenus  118756923   NaN   greek   
4  cf005       P18056         Irenäus von Lyon  118555766   NaN   greek   

    ogl_id pta_id Unnamed: 8  
0  tlg1443    NaN        NaN  
1  tlg1622    NaN        NaN  
2  tlg1205    NaN        NaN  
3  tlg1725    NaN        NaN  
4  tlg1447    NaN        NaN  


In [29]:
TEI_NS  = "http://www.tei-c.org/ns/1.0"
NS_MAP  = {"tei": TEI_NS}

def _text(element):
    """Return stripped text from an lxml element, or empty string."""
    if element is None:
        return ""
    return " ".join((element.text or "").split())


def extract_tei_metadata(xml_path: Path) -> dict:
    """
    Parse a TEI XML file and extract:
      - title   : //teiHeader/fileDesc/titleStmt/title
      - editor  : //teiHeader/fileDesc/titleStmt/editor  (first occurrence)
      - language: //teiHeader/profileDesc/langUsage/language/@ident
                  fallback → //text/@xml:lang
                  fallback → //TEI/@xml:lang
    """
    result = {"title": "", "editor": "", "language": ""}

    try:
        tree = etree.parse(str(xml_path))
        root = tree.getroot()

        # Detect whether TEI namespace is used
        ns = TEI_NS if root.tag.startswith("{") else ""
        pfx = "tei:" if ns else ""
        nm  = NS_MAP if ns else {}

        def xp(xpath_expr):
            """Run an XPath, auto-inserting the tei: prefix."""
            expr = xpath_expr.replace("tei:", pfx)
            return root.xpath(expr, namespaces=nm)

        # ── Title ────────────────────────────────────────────────────────────
        # Try the most specific path first, then progressively loosen
        for title_xpath in [
            ".//tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title[@type='main']",
            ".//tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title",
            ".//tei:titleStmt/tei:title",
            ".//tei:title",
        ]:
            hits = xp(title_xpath)
            if hits:
                result["title"] = _text(hits[0])
                break

        # ── Editor ───────────────────────────────────────────────────────────
        for editor_xpath in [
            ".//tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:editor",
            ".//tei:titleStmt/tei:editor",
            ".//tei:editor",
        ]:
            hits = xp(editor_xpath)
            if hits:
                # Collect all editors, join with '; '
                editors = [_text(e) for e in hits if _text(e)]
                if editors:
                    result["editor"] = "; ".join(editors)
                    break

        # ── Language ─────────────────────────────────────────────────────────
        # 1) langUsage/language @ident
        lang_nodes = xp(".//tei:profileDesc/tei:langUsage/tei:language")
        if lang_nodes:
            idents = [n.get("ident") or n.get("xml:lang") or n.get("{http://www.w3.org/XML/1998/namespace}lang") or _text(n)
                      for n in lang_nodes]
            idents = [i for i in idents if i]
            if idents:
                result["language"] = "; ".join(idents)

        # 2) fallback: @xml:lang on <text> element
        if not result["language"]:
            XML_NS = "http://www.w3.org/XML/1998/namespace"
            text_nodes = xp(".//tei:text") or xp(".//tei:TEI") or [root]
            for node in text_nodes:
                lang = node.get(f"{{{XML_NS}}}lang") or node.get("xml:lang")
                if lang:
                    result["language"] = lang
                    break

    except etree.XMLSyntaxError as exc:
        print(f"  ⚠ XML parse error in {xml_path.name}: {exc}")

    return result

In [30]:
data_root = Path(DATA_DIR)

records = []
unmatched_folders = set()
parse_errors = []

xml_files = sorted(data_root.rglob("*.xml"))
print(f"Found {len(xml_files)} XML files under '{DATA_DIR}/'\n")

for xml_path in xml_files:
    # Derive the subfolder name (immediate child of DATA_DIR)
    # Works even for deeply nested files — uses the first subfolder level
    rel = xml_path.relative_to(data_root)
    folder_key = rel.parts[0].strip().lower() if len(rel.parts) > 1 else "__root__"

    # Lookup church father name
    church_father = lookup.get(folder_key)
    if church_father is None:
        unmatched_folders.add(folder_key)
        church_father = f"[unknown: {rel.parts[0]}]"  # keep original casing for hint

    # Extract TEI metadata
    meta = extract_tei_metadata(xml_path)

    records.append({
        "church_father": church_father,
        "work":          meta["title"],
        "language":      meta["language"],
        "source":        xml_path.name,          # just the filename
        # ── bonus columns (hidden by default; comment out if unwanted) ──
        "_editor":       meta["editor"],
        "_folder":       rel.parts[0],
        "_full_path":    str(xml_path),
    })

print(f"Processed {len(records)} files.")

if unmatched_folders:
    print(f"\n⚠ {len(unmatched_folders)} folder key(s) not found in lookup CSV:")
    for f in sorted(unmatched_folders):
        print(f"   • {f}")
    print("  → Add these keys to church-fathers.csv to resolve.")

Found 700 XML files under '../data/'

  ⚠ XML parse error in Chrysostom_In_Joannem.xml: Extra content at the end of the document, line 5, column 1 (Chrysostom_In_Joannem.xml, line 5)
Processed 700 files.


In [31]:
df = pd.DataFrame(records)

# output columns 
output_cols = ["church_father", "work", "language", "source", "_editor"]

df_out = df[output_cols].copy()

print(f"Shape: {df_out.shape}")
print(f"\nMissing values:")
print(df_out.isnull().sum())
print(f"\nEmpty strings:")
print((df_out == "").sum())

df_out.head(10)

Shape: (700, 5)

Missing values:
church_father    0
work             0
language         0
source           0
_editor          0
dtype: int64

Empty strings:
church_father      0
work               2
language          17
source             0
_editor          142
dtype: int64


,church_father,work,language,source,_editor
0,Ignatius von Antiochien,Epistulae vii genuinae,grc; lat,tlg1443.tlg001.1st1K-grc1.xml,Kirsopp Lake
1,Ignatius von Antiochien,Epistulae interpolatae et epistulae suppositic...,grc; lat,tlg1443.tlg002.1st1K-grc1.xml,Francis Xavier Funk; Francis Diekamp
2,Polycarpus Smyrnäus,Epistula ad Philippenses,grc; lat,tlg1622.tlg001.1st1K-grc1.xml,Kirsopp Lake
3,Athenagoras Atheniensis,Legatio sive Supplicatio pro Christianis,grc; lat,tlg1205.tlg001.perseus-grc1.xml,Johann Karl Theodor von Otto
4,Athenagoras Atheniensis,Supplication pro Christianis,grc; lat,tlg1205.tlg001.perseus-grc2.xml,Francis Andrew March; William Baxter Owen
5,Athenagoras Atheniensis,De Resurrectione,grc; lat,tlg1205.tlg002.perseus-grc1.xml,Johann Karl Theodor von Otto
6,Theophilus Antiochenus,Ad Autolycum,grc; lat,tlg1725.tlg001.perseus-grc1.xml,William Gilson Humphry
7,Irenäus von Lyon,Libros quinque adversus haereses,grc; lat,tlg1447.tlg001.1st1K-grc1.xml,William Wigan Harvey
8,Irenäus von Lyon,Fragmenta Synodicae Epistulae,grc; lat,tlg1447.tlg009.1st1K-grc1.xml,Martin Joseph Routh
9,Quintus Septimius Florens Tertullian,Responsa ex libris Digestorum,lat,001_Auctor-incertus_Responsa-ex-libris-Digesto...,Jacques-Paul Migne


In [32]:
# stats
print(f"Total files      : {len(df_out)}")
print(f"Church fathers   : {df_out['church_father'].nunique()}")
print()
print("Files per church father:")
print(df_out['church_father'].value_counts().to_string())
print()
print("Language distribution:")
print(df_out['language'].value_counts().to_string())

Total files      : 700
Church fathers   : 52

Files per church father:
church_father
Augustinus von Hippo                               155
Sophronius Eusebius Hieronymus                     104
Ambrosius von Mailand                               49
Quintus Septimius Florens Tertullian                37
Anicius Manlius Severinus Boethius                  32
Papst Gregor I. (der Große)                         26
Isidor von Sevilla                                  26
Prosper von Aquitanien                              25
Cyprian                                             24
Eusebius von Caesarea                               24
Hilarius von Poitiers                               23
Lucius Caecilius Firmianus Lactantius (Laktanz)     14
Claudius Gordianus Fulgentius                       14
Leo I. (der Große)                                  13
Gaius Marius Victorinus                             10
Rufinus von Aquileia                                10
Origenes                           

In [33]:
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")  # utf-8-sig for Excel compat